# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Prepare the dataset

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [3]:
from lightningrod.training import prepare_for_training
from lightningrod.training.samples import BinaryAnswerType

default_dataset_id = "e87e04c3-4c0d-49ab-97bf-b30d724395d3" # paste it here, or set it as an environment variable
dataset_id = config.get_config_value("DATASET_ID", default_dataset_id)

dataset = lr.datasets.get(dataset_id)
dataset.download()

[Sample(id='00b0559f-9722-488a-ae06-6c0ebd169a1f', seed=Seed(seed_text="Title: Rinse\nOne-liner: We're building the One Medical for dental\n\nDescription: Rinse is building the One Medical for dental. Beautiful studios, easy to book same day appointments, and anxiety-free care focused on prevention, not the upsell.\n\nWebsite: http://www.rinse.dental\nYC URL: https://www.ycombinator.com/companies/rinse\nBatch: Summer 2021\nIndustry: Healthcare\nTags: Consumer Health Services, Digital Health, Telemedicine, Primary Care", url=None, seed_creation_date=datetime.datetime(2021, 7, 15, 15, 55, 27), search_query=None, additional_properties={}), question=ForwardLookingQuestion(question_text='Will Rinse (rinse.dental) be an active business entity with at least one operational physical dental studio on January 1, 2026?', date_close=datetime.datetime(2026, 1, 1, 0, 0), event_date=datetime.datetime(2021, 7, 15, 15, 55, 27), resolution_criteria="The question resolves to 'Yes' if on January 1, 2026, 

In [4]:
import pandas as pd

PROMPT_TEMPLATE = """CONTEXT (original post):

{seed_text}

QUESTION:
{question_text}

TODAY'S DATE:
{question_date}

RESOLUTION CRITERIA:
{resolution_criteria}

CLOSE DATE:
{date_close}

ANSWER FORMAT:
{answer_instructions}"""

train, test = prepare_for_training(
    samples=dataset.samples(),
    answer_type=BinaryAnswerType(),
    test_size=0.2,
    days_to_resolution_range=(90, None),
    deduplicate_key_fn=lambda sample: (sample.question.question_text, sample.seed.seed_text, sample.label.resolution_date),
    prompt_template=PROMPT_TEMPLATE,
    filter_leaky_train=False,
    verbose=True,
)

display(pd.DataFrame(train).head())
display(pd.DataFrame(test).head())

[prepare_for_training] Starting with 500 samples
[filter] Dropped 89 invalid, 7 horizon → 404 remain
[dedup] 404 remain (0 duplicates)
[split] Temporal split: 323 train, 81 test


,sample_id,prompt,correct_answer,answer_type,reward_function_type,answer_parser_type
0,689807e4-34b8-4fc6-8ffa-f94e50b741ca,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary
1,7d3edc97-21e6-46ef-9516-21381e9d33c5,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary
2,ba1e44ee-2638-4763-acdf-e9231bbf9b62,"[{'role': 'user', 'content': 'CONTEXT (origina...",0,binary,binary_log_score,binary
3,98bd1556-957c-46da-a315-2662723316b5,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary
4,dc5f3568-7f6f-4364-8432-e4923d8b8d8f,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary


,sample_id,prompt,correct_answer,answer_type,reward_function_type,answer_parser_type
0,f2bbc53e-c2cd-485d-8486-e34ced62a502,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary
1,b727a00c-7b76-4910-9e2c-04c11dbf3348,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary
2,33375f38-6298-4552-97a6-1f1ceb612883,"[{'role': 'user', 'content': 'CONTEXT (origina...",1,binary,binary_log_score,binary
3,5846c557-45b8-4f0a-b274-43fa2abc14f9,"[{'role': 'user', 'content': 'CONTEXT (origina...",0,binary,binary_log_score,binary
4,cb08a2af-93f8-4305-8284-6831931c1a6c,"[{'role': 'user', 'content': 'CONTEXT (origina...",0,binary,binary_log_score,binary


## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [5]:
from lightningrod.training import TrainingConfig, SampleDatasetConfig

config = TrainingConfig(
    base_model="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=50,
    dataset=SampleDatasetConfig(
        id=dataset_id,
        sample_ids=list(map(lambda s: s["sample_id"], train)),
    ),
)

cost_estimate = lr.training.estimate_cost(config)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.10
Effective steps: 11
Train tokens: 315,316
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display. Use this when you want to wait for the job to finish in the notebook. Skip this if you used `create` above and prefer to poll manually.

In [14]:
job = lr.training.run(config, name="Forecasting fine-tune")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: Forecasting fine-tune                                                                                   │
│                                                                                                                 │
│    Reward: latest -0.3970  avg -0.7675  (11 steps)  (higher is better)                                          │
│                                                                                                                 │
│    Cost:  $0.06                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job b1f60057-f348-4d86-a3cf-eedb69ee603a completed with status: COMPLETED
Trained model ID: checkpoint:b1f60057-f348-4d86-a3cf-eedb69ee603a


## List and get jobs

List all training jobs or fetch a specific job by ID.

In [15]:
import pandas as pd

jobs_response = lr.training.list(limit=5)

latest_model_id = jobs_response.jobs[0].model_id

df = pd.DataFrame([
    {
        "Job ID": j.id,
        "Status": j.status,
        "Base Model": getattr(j.config, "base_model", None),
        "Trained Model ID": j.model_id,
        "Date": j.created_at,
        "Cost": j.cost_dollars,
    }
    for j in jobs_response.jobs
])

df

,Job ID,Status,Base Model,Trained Model ID,Date,Cost
0,b1f60057-f348-4d86-a3cf-eedb69ee603a,COMPLETED,Qwen/Qwen3-4B-Instruct-2507,checkpoint:b1f60057-f348-4d86-a3cf-eedb69ee603a,2026-03-13 11:23:14.431000+00:00,0.058802
1,fba989dd-971f-45ac-850d-c17c9a294c58,COMPLETED,Qwen/Qwen3-4B-Instruct-2507,checkpoint:fba989dd-971f-45ac-850d-c17c9a294c58,2026-03-13 11:10:19.089000+00:00,0.057982
2,d11d8e03-a464-4058-a9f7-eba0d6aa8036,RUNNING,Qwen/Qwen3-4B-Instruct-2507,None,2026-03-13 11:07:14.592000+00:00,NaN
3,4d815d9d-3efd-40ef-888c-39291c1f4b75,RUNNING,Qwen/Qwen3-4B-Instruct-2507,None,2026-03-13 11:04:31.141000+00:00,NaN
4,61a6919e-802f-4ec8-b5a6-babc2412b233,RUNNING,Qwen/Qwen3-4B-Instruct-2507,None,2026-03-13 11:00:39.200000+00:00,NaN


## Inference with your trained model

Once training completes, use `job.model_id` with the OpenAI-compatible API. We also have a pre-trained foresight-v3 model for forecasting — see [08_foresight_model.ipynb](08_foresight_model.ipynb).

In [16]:
%pip install openai
from IPython.display import clear_output
clear_output()

from openai import OpenAI
from lightningrod.utils import config

base_url = config.get_config_value("LIGHTNINGROD_BASE_URL", "https://api.lightningrod.ai/api/public/v1")
client = OpenAI(api_key=api_key, base_url=f"{base_url}/openai")

In [ ]:
response = client.chat.completions.create(
    model=latest_model_id,
    messages=[
        {"role": "system", "content": "Answer as a probability between 0 and 1 between <answer></answer> tags."},
        {"role": "user", "content": "Will the Fed cut rates by 25bp in March 2026?"}
    ]
)
print(response.choices[0].message.content)

<answer>0.35</answer>


## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset and reports metrics. Use the same dataset for a quick check, or a separate test split for production.

In [6]:
eval_job = lr.evals.run(
    # model_id=latest_model_id,
    model_id="checkpoint:b1f60057-f348-4d86-a3cf-eedb69ee603a",
    dataset=SampleDatasetConfig(
        id=dataset_id,
        sample_ids=list(map(lambda s: s["sample_id"], test)),
    ),
    benchmark_model_id="openai/gpt-5.2",
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: 5350a6e8-87b8-4c3d-94f2-b46986a66349                                                                     │
│    Model: checkpoint:b1f60057-f348-4d86-a3cf-eedb69ee603a                                                       │
│    Dataset: e87e04c3-4c0d-49ab-97bf-b30d724395d3                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┓                                                        │
│  ┃ Metric              ┃    base ┃ trained ┃ benchmark ┃                                                        │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━┩                                                        │
│  │ brier_score         │  0.1929 │  0.1871 │    0.2005 │                                                        │
│  │ ece                 │  0.0677 │  0.0899 │    0.0943 │                                                        │
│  │ extra               │      {} │      {} │        {} │                                                        │
│  │ mean_reward         │ -0.5724 │ -0.5586 │   -0.5892 │                                                        │
│  │ mean_valid_reward   │ -0.5724 │ -0.5586 │   -0.5892 │                                                        │
│  │ n_samples           │      81 │      81 │        81 │                                                        │
│  │ n_valid             │      81 │      81 │        81 │                                                        │
│  │ parse_rate          │  1.0000 │  1.0000 │    1.0000 │                                                        │
│  │ total_cost          │  0.0016 │  0.0016 │         — │                                                        │
│  │ total_input_tokens  │   20154 │   20154 │     18641 │                                                        │
│  │ total_output_tokens │     788 │     806 │     14844 │                                                        │
│  └─────────────────────┴─────────┴─────────┴───────────┘                                                        │
│                                                                                                                 │
│    Cost:  $0.00                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
import pandas as pd

evals_response = lr.evals.list(limit=5)
pd.DataFrame([
    {
        "Eval ID": e.id,
        "Status": e.status,
        "Error": e.error_message,
        "Dataset": e.config.dataset.id,
        "Date": e.created_at,
    }
    for e in evals_response.jobs
])

,Eval ID,Status,Error,Dataset,Date
0,5350a6e8-87b8-4c3d-94f2-b46986a66349,COMPLETED,None,e87e04c3-4c0d-49ab-97bf-b30d724395d3,2026-03-13 12:26:13.004000+00:00
1,e95940a1-214b-4a67-9a25-24ddf74f6909,FAILED,Type <class 'lightningrodlabs.v2.analysis.anal...,e87e04c3-4c0d-49ab-97bf-b30d724395d3,2026-03-13 12:16:50.123000+00:00
2,8673c899-f159-402d-be1c-15117f00b814,FAILED,Type <class 'lightningrodlabs.v2.analysis.anal...,e87e04c3-4c0d-49ab-97bf-b30d724395d3,2026-03-13 12:11:02.917000+00:00
3,17e16095-02f5-4c55-9163-188f0ff95037,COMPLETED,None,bart/training-demo,2026-03-10 19:15:55.766000+00:00
4,04b3e27f-c20a-4e1b-b563-91f9897d034b,COMPLETED,None,bart/training-demo,2026-03-09 21:56:14.858000+00:00


In [11]:
from lightningrod.training import print_eval

latest_eval_id = evals_response.jobs[0].id

eval_job = lr.evals.get(latest_eval_id)
print_eval(eval_job)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: 5350a6e8-87b8-4c3d-94f2-b46986a66349                                                                     │
│    Model: checkpoint:b1f60057-f348-4d86-a3cf-eedb69ee603a                                                       │
│    Dataset: e87e04c3-4c0d-49ab-97bf-b30d724395d3                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┓                                                        │
│  ┃ Metric              ┃    base ┃ trained ┃ benchmark ┃                                                        │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━┩                                                        │
│  │ brier_score         │  0.1929 │  0.1871 │    0.2005 │                                                        │
│  │ ece                 │  0.0677 │  0.0899 │    0.0943 │                                                        │
│  │ extra               │      {} │      {} │        {} │                                                        │
│  │ mean_reward         │ -0.5724 │ -0.5586 │   -0.5892 │                                                        │
│  │ mean_valid_reward   │ -0.5724 │ -0.5586 │   -0.5892 │                                                        │
│  │ n_samples           │      81 │      81 │        81 │                                                        │
│  │ n_valid             │      81 │      81 │        81 │                                                        │
│  │ parse_rate          │  1.0000 │  1.0000 │    1.0000 │                                                        │
│  │ total_cost          │  0.0016 │  0.0016 │         — │                                                        │
│  │ total_input_tokens  │   20154 │   20154 │     18641 │                                                        │
│  │ total_output_tokens │     788 │     806 │     14844 │                                                        │
│  └─────────────────────┴─────────┴─────────┴───────────┘                                                        │
│                                                                                                                 │
│    Cost:  $0.00                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.